# Causal Intervention Analysis

Tests native functional dependence of the scratch Transformer's predictive distribution on state-defining inputs and on counterfactual internal activations.

For each Markov order, seed, and sufficient-context condition, the analysis compares factual predictions with counterfactual predictions induced by changing a state-defining token. At \(L=k+1\), an additional intervention on the token outside the true Markov state provides an irrelevant-token control.

Layer-wise activation patching is included as a positive control showing whether counterfactual residual-stream activations can steer the native predictive distribution. These interventions establish functional dependence under the specified manipulations; they do not identify a unique internal state code or mechanistic circuit.


In [ ]:
import os, sys, json, math, random, platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_BF16 = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print("AMP:", "BF16" if USE_BF16 else ("FP16" if torch.cuda.is_available() else "OFF"))

In [ ]:
# ============================================================
# PATHS — existing Part 1 is READ ONLY
# ============================================================

PART1_ROOT = Path.cwd() / "Markov_Context_Paper" / "Part1"
RUNS_DIR = PART1_ROOT / "runs"

OUT_DIR = PART1_ROOT / "causal_intervention"
CSV_DIR = OUT_DIR / "csv"
FIG_DIR = OUT_DIR / "figures"

for p in [OUT_DIR, CSV_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

assert RUNS_DIR.exists(), f"Missing runs directory: {RUNS_DIR}"

print("READ ONLY runs:", RUNS_DIR)
print("NEW outputs:", OUT_DIR)

In [ ]:
# ============================================================
# EXPERIMENT CONFIGURATION
# ============================================================

V = 8
KS = [2, 3, 4]
SEEDS = [0, 1, 2, 3, 4]

EXPERIMENT_BY_K = {
    2: "main_recovery",
    3: "main_recovery",
    4: "k4_refinement_12k",
}

N_CONTEXTS = 20_000
EVAL_BATCH = 2_000
CHAIN_WARMUP = 256

BOOTSTRAP_REPS = 5_000

# Internal activation-patching stage (L=k+1 only).
RUN_ACTIVATION_PATCHING = True
PATCH_N_CONTEXTS = 5_000
PATCH_BATCH = 500
BOOTSTRAP_SEED = 123
EPS = 1e-8

QUICK_TEST = False
if QUICK_TEST:
    N_CONTEXTS = 2_000
    EVAL_BATCH = 500
    PATCH_N_CONTEXTS = 1_000
    PATCH_BATCH = 250

print("Models expected:", len(KS) * 2 * len(SEEDS))
print("Contexts/model:", N_CONTEXTS)

## Core definitions

These definitions reproduce the Part 1 scratch model and persistent Markov sampler exactly enough to load the saved state dictionaries and kernels. No optimizer or training code is used.

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def context_to_index(context, V):
    idx = torch.zeros(context.shape[:-1], dtype=torch.long, device=context.device)
    for j in range(context.shape[-1]):
        idx = idx * V + context[..., j]
    return idx


class PersistentMarkovSampler:
    def __init__(self, P, V, k, batch_size, max_history, seed, warmup_steps=64):
        self.P = P
        self.V = V
        self.k = k
        self.batch_size = batch_size
        self.max_history = max(max_history, k)
        self.gen = torch.Generator(device=DEVICE)
        self.gen.manual_seed(seed + 4_000_003)
        self.history = torch.randint(
            0, V, (batch_size, self.max_history),
            generator=self.gen, device=DEVICE
        )
        for _ in range(warmup_steps):
            self.advance()

    @torch.no_grad()
    def draw_next(self):
        full_k = self.history[:, -self.k:]
        idx = context_to_index(full_k, self.V)
        probs = self.P[idx]
        return torch.multinomial(probs, 1, generator=self.gen).squeeze(1)

    @torch.no_grad()
    def advance(self):
        y = self.draw_next()
        self.history = torch.cat([self.history[:, 1:], y[:, None]], dim=1)
        return y

    @torch.no_grad()
    def sample(self, L):
        x = self.history[:, -L:].clone()
        y = self.draw_next()
        self.history = torch.cat([self.history[:, 1:], y[:, None]], dim=1)
        return x, y


class TinyCausalTransformer(nn.Module):
    def __init__(self, V, max_context, d_model=256, n_layers=3, n_heads=8, dim_ff=512):
        super().__init__()
        self.token_emb = nn.Embedding(V, d_model)
        self.pos_emb = nn.Embedding(max_context, d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=dim_ff,
            dropout=0.0,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, V)
        self.max_context = max_context

    def forward(self, x):
        B, L = x.shape
        pos = torch.arange(L, device=x.device)[None, :]
        h = self.token_emb(x) + self.pos_emb(pos)
        mask = torch.triu(
            torch.ones((L, L), dtype=torch.bool, device=x.device),
            diagonal=1
        )
        h = self.encoder(h, mask=mask)
        return self.head(self.norm(h[:, -1, :]))

## Run discovery and checkpoint loading

The notebook discovers runs from their saved `config.json` rather than reconstructing hashed run IDs. It requires exactly one matching completed run for each requested \((k,L,\mathrm{seed})\).

In [ ]:
def trusted_torch_load(path, map_location):
    # Checkpoints/kernels are self-generated project artifacts.
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


# Exact training configuration used in Part1_Markov_Context_Sufficiency_A100_v2.
# We match these fields so stale QUICK_TEST / alternate runs are never selected.
EXPECTED_CFG = {
    "main_recovery": {
        "steps": 3000,
        "batch_size": 1024,
        "lr": 1e-3,
        "weight_decay": 1e-2,
        "d_model": 256,
        "n_layers": 3,
        "n_heads": 8,
        "dim_ff": 512,
        "eval_batch": 4096,
        "save_every": 500,
        "sparse_s": None,
    },
    "k4_refinement_12k": {
        "steps": 12000,
        "batch_size": 1024,
        "lr": 1e-3,
        "weight_decay": 1e-2,
        "d_model": 256,
        "n_layers": 3,
        "n_heads": 8,
        "dim_ff": 512,
        "eval_batch": 4096,
        "save_every": 500,
        "sparse_s": None,
    },
}


def _cfg_matches_expected(cfg, experiment):
    exp = EXPECTED_CFG[experiment]
    for key, expected in exp.items():
        got = cfg.get(key, None)
        if isinstance(expected, float):
            try:
                if not math.isclose(float(got), expected, rel_tol=1e-12, abs_tol=1e-12):
                    return False
            except Exception:
                return False
        else:
            if got != expected:
                return False
    return True


def _canonical_run_name(cfg):
    # Mirrors the original Part1 RunConfig.run_id() exactly.
    import hashlib
    s = json.dumps(cfg, sort_keys=True, default=str).encode()
    h = hashlib.sha256(s).hexdigest()[:12]
    sp = cfg.get("sparse_s") or 0
    return (
        f"{cfg['experiment']}_V{cfg['V']}_k{cfg['k']}_L{cfg['L']}_seed{cfg['seed']}"
        f"_sp{sp}_d{cfg['d_model']}_ly{cfg['n_layers']}_{h}"
    )


def discover_run(experiment, V, k, L, seed):
    root = RUNS_DIR / experiment
    if not root.exists():
        raise FileNotFoundError(f"Experiment directory not found: {root}")

    broad_matches = []
    exact_matches = []

    for cfg_path in root.glob("*/config.json"):
        try:
            cfg = json.loads(cfg_path.read_text(encoding="utf-8"))
        except Exception:
            continue

        same_identity = (
            int(cfg.get("V", -1)) == int(V)
            and int(cfg.get("k", -1)) == int(k)
            and int(cfg.get("L", -1)) == int(L)
            and int(cfg.get("seed", -1)) == int(seed)
            and cfg.get("experiment") == experiment
        )
        if not same_identity:
            continue

        broad_matches.append((cfg_path.parent, cfg))
        if _cfg_matches_expected(cfg, experiment):
            exact_matches.append((cfg_path.parent, cfg))

    if not exact_matches:
        details = [
            {
                "dir": str(rd),
                "steps": cfg.get("steps"),
                "batch_size": cfg.get("batch_size"),
                "d_model": cfg.get("d_model"),
                "n_layers": cfg.get("n_layers"),
            }
            for rd, cfg in broad_matches
        ]
        raise RuntimeError(
            f"No run matches the ORIGINAL Part1 config for experiment={experiment}, "
            f"V={V}, k={k}, L={L}, seed={seed}. "
            f"Identity-matched candidates: {details}"
        )

    # Prefer the canonical directory name generated by the original run_id().
    canonical = [
        (rd, cfg) for rd, cfg in exact_matches
        if rd.name == _canonical_run_name(cfg)
    ]

    candidates = canonical if canonical else exact_matches

    # Only accept candidates that contain the immutable source kernel and a checkpoint.
    complete = []
    for rd, cfg in candidates:
        has_kernel = (rd / "kernel.pt").exists()
        has_ckpt = (rd / "checkpoint_best.pt").exists() or (rd / "checkpoint_last.pt").exists()
        if has_kernel and has_ckpt:
            complete.append((rd, cfg))

    if len(complete) == 1:
        if len(broad_matches) > 1:
            print(
                f"[duplicate-safe] {experiment} V={V} k={k} L={L} seed={seed}: "
                f"{len(broad_matches)} identity matches -> using canonical {complete[0][0].name}"
            )
        return complete[0]

    raise RuntimeError(
        f"Could not select one canonical complete run for experiment={experiment}, "
        f"V={V}, k={k}, L={L}, seed={seed}. "
        f"broad={len(broad_matches)}, exact={len(exact_matches)}, "
        f"canonical={len(canonical)}, complete={len(complete)}. "
        f"Candidates={[str(rd) for rd, _ in candidates]}"
    )


def choose_checkpoint(run_dir):
    best = run_dir / "checkpoint_best.pt"
    last = run_dir / "checkpoint_last.pt"
    if best.exists():
        return best
    if last.exists():
        return last
    raise FileNotFoundError(f"No checkpoint found in {run_dir}")


def load_model_and_kernel(experiment, V, k, L, seed):
    run_dir, cfg = discover_run(experiment, V, k, L, seed)
    ckpt_path = choose_checkpoint(run_dir)
    kernel_path = run_dir / "kernel.pt"

    if not kernel_path.exists():
        raise FileNotFoundError(f"Missing exact saved kernel: {kernel_path}")

    P = trusted_torch_load(kernel_path, map_location=DEVICE)
    if isinstance(P, dict):
        raise TypeError(f"Unexpected kernel payload in {kernel_path}")
    P = P.float().to(DEVICE)

    ck = trusted_torch_load(ckpt_path, map_location=DEVICE)
    state = ck["model"] if isinstance(ck, dict) and "model" in ck else ck

    max_context = int(state["pos_emb.weight"].shape[0])
    d_model = int(cfg.get("d_model", state["token_emb.weight"].shape[1]))
    n_layers = int(cfg.get("n_layers", 3))
    n_heads = int(cfg.get("n_heads", 8))
    dim_ff = int(cfg.get("dim_ff", 512))

    model = TinyCausalTransformer(
        V=V,
        max_context=max_context,
        d_model=d_model,
        n_layers=n_layers,
        n_heads=n_heads,
        dim_ff=dim_ff,
    ).to(DEVICE)

    model.load_state_dict(state, strict=True)
    model.eval()

    return model, P, cfg, run_dir, ckpt_path


inventory = []
for k in KS:
    exp = EXPERIMENT_BY_K[k]
    for L in [k, k + 1]:
        for seed in SEEDS:
            run_dir, cfg = discover_run(exp, V, k, L, seed)
            ckpt = choose_checkpoint(run_dir)
            kernel = run_dir / "kernel.pt"
            inventory.append({
                "experiment": exp,
                "k": k,
                "L": L,
                "seed": seed,
                "steps": cfg.get("steps"),
                "run_dir": str(run_dir),
                "checkpoint": str(ckpt),
                "kernel_exists": kernel.exists(),
            })

inventory_df = pd.DataFrame(inventory)
display(inventory_df)
assert inventory_df["kernel_exists"].all(), "At least one kernel.pt is missing."
assert len(inventory_df) == 30, f"Expected 30 models, found {len(inventory_df)}."
print("Preflight OK:", len(inventory_df), "canonical models ready.")


## Counterfactual construction and metrics

For an input \(x=(x_{t-L+1},\ldots,x_t)\):

- the true order-\(k\) state occupies positions `L-k ... L-1`;
- the **critical position** is therefore `L-k`, the oldest token still inside the state;
- when \(L=k+1\), position `0` is outside the Markov state and is the **irrelevant control**.

The replacement token is sampled uniformly from the other \(V-1\) symbols, deterministically from the evaluation seed.

The oracle distribution is always computed from the exact saved transition kernel, not from sampled labels.

In [ ]:
def make_alternative_tokens(original, V, generator):
    # Uniformly choose one of the V-1 symbols different from original.
    r = torch.randint(
        low=0, high=V - 1, size=original.shape,
        generator=generator, device=original.device
    )
    return (original + 1 + r) % V


def oracle_probs(P, x, V, k):
    state = x[:, -k:]
    idx = context_to_index(state, V)
    return P[idx].float()


def kl_pq(p, q, eps=1e-8):
    p = p.float().clamp_min(eps)
    q = q.float().clamp_min(eps)
    return (p * (p.log() - q.log())).sum(dim=-1)


def js_divergence(p, q, eps=1e-8):
    p = p.float().clamp_min(eps)
    q = q.float().clamp_min(eps)
    m = 0.5 * (p + q)
    return 0.5 * kl_pq(p, m, eps) + 0.5 * kl_pq(q, m, eps)


def entropy(p, eps=1e-8):
    p = p.float().clamp_min(eps)
    return -(p * p.log()).sum(dim=-1)


@torch.no_grad()
def model_probs(model, x):
    with torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
        enabled=torch.cuda.is_available()
    ):
        logits = model(x)
    return torch.softmax(logits.float(), dim=-1)


def summarize_array(v):
    v = np.asarray(v, dtype=float)
    return {
        "mean": float(np.mean(v)),
        "sd": float(np.std(v, ddof=1)) if len(v) > 1 else 0.0,
        "median": float(np.median(v)),
        "q25": float(np.quantile(v, 0.25)),
        "q75": float(np.quantile(v, 0.75)),
    }

In [ ]:
@torch.no_grad()
def evaluate_intervention_run(model, P, V, k, L, source_seed, n_contexts, batch_size):
    # Inferential unit later = trained model/source seed, not individual contexts.
    sampler_seed = 8_100_000 + 10_000 * k + 100 * L + source_seed
    intervention_seed = 9_200_000 + 10_000 * k + 100 * L + source_seed

    sampler = PersistentMarkovSampler(
        P=P, V=V, k=k, batch_size=batch_size,
        max_history=max(L, k),
        seed=sampler_seed, warmup_steps=CHAIN_WARMUP
    )

    gen = torch.Generator(device=DEVICE)
    gen.manual_seed(intervention_seed)

    critical_rows = []
    irrelevant_rows = []

    seen = 0
    while seen < n_contexts:
        b = min(batch_size, n_contexts - seen)

        if b == batch_size:
            x, _ = sampler.sample(L)
        else:
            tmp = PersistentMarkovSampler(
                P=P, V=V, k=k, batch_size=b,
                max_history=max(L, k),
                seed=sampler_seed + seen + 17,
                warmup_steps=CHAIN_WARMUP
            )
            x, _ = tmp.sample(L)

        p_before = oracle_probs(P, x, V, k)
        q_before = model_probs(model, x)

        # Critical intervention: oldest token inside the k-token state.
        critical_pos = L - k
        x_crit = x.clone()
        x_crit[:, critical_pos] = make_alternative_tokens(
            x[:, critical_pos], V, gen
        )

        p_crit = oracle_probs(P, x_crit, V, k)
        q_crit = model_probs(model, x_crit)

        kl_target_before = kl_pq(p_crit, q_before)
        kl_target_after = kl_pq(p_crit, q_crit)
        tracking_gain = kl_target_before - kl_target_after

        critical_rows.append(pd.DataFrame({
            "tracking_gain": tracking_gain.cpu().numpy(),
            "kl_target_before": kl_target_before.cpu().numpy(),
            "kl_target_after": kl_target_after.cpu().numpy(),
            "oracle_shift_js": js_divergence(p_before, p_crit).cpu().numpy(),
            "model_shift_js": js_divergence(q_before, q_crit).cpu().numpy(),
            "factual_kl_before": kl_pq(p_before, q_before).cpu().numpy(),
            "target_oracle_entropy": entropy(p_crit).cpu().numpy(),
        }))

        # Irrelevant control: at L=k+1, position 0 is outside the state.
        if L > k:
            x_irrel = x.clone()
            x_irrel[:, 0] = make_alternative_tokens(x[:, 0], V, gen)

            p_irrel = oracle_probs(P, x_irrel, V, k)
            q_irrel = model_probs(model, x_irrel)

            max_oracle_diff = (p_irrel - p_before).abs().max().item()
            if max_oracle_diff > 1e-7:
                raise AssertionError(
                    f"Irrelevant intervention changed oracle: max diff={max_oracle_diff}"
                )

            kl_same_before = kl_pq(p_before, q_before)
            kl_same_after = kl_pq(p_before, q_irrel)
            tracking_gain_irrel = kl_same_before - kl_same_after

            irrelevant_rows.append(pd.DataFrame({
                "tracking_gain": tracking_gain_irrel.cpu().numpy(),
                "kl_target_before": kl_same_before.cpu().numpy(),
                "kl_target_after": kl_same_after.cpu().numpy(),
                "oracle_shift_js": js_divergence(p_before, p_irrel).cpu().numpy(),
                "model_shift_js": js_divergence(q_before, q_irrel).cpu().numpy(),
                "factual_kl_before": kl_same_before.cpu().numpy(),
                "target_oracle_entropy": entropy(p_before).cpu().numpy(),
            }))

        seen += b

    def aggregate(frames, intervention):
        d = pd.concat(frames, ignore_index=True)
        row = {
            "intervention": intervention,
            "n_contexts": len(d),
        }
        for col in d.columns:
            s = summarize_array(d[col].values)
            for stat, val in s.items():
                row[f"{col}_{stat}"] = val
        row["positive_tracking_fraction"] = float((d["tracking_gain"] > 0).mean())
        return row

    out = [aggregate(critical_rows, "critical")]
    if irrelevant_rows:
        out.append(aggregate(irrelevant_rows, "irrelevant"))

    return pd.DataFrame(out)

## Run the causal experiment

This cell performs inference only. It never writes to the existing run directories.

In [ ]:
all_rows = []

for k in KS:
    experiment = EXPERIMENT_BY_K[k]

    for L in [k, k + 1]:
        for seed in SEEDS:
            print(f"RUN k={k} L={L} seed={seed} [{experiment}]")

            model, P, cfg, run_dir, ckpt_path = load_model_and_kernel(
                experiment=experiment, V=V, k=k, L=L, seed=seed
            )

            run_df = evaluate_intervention_run(
                model=model,
                P=P,
                V=V,
                k=k,
                L=L,
                source_seed=seed,
                n_contexts=N_CONTEXTS,
                batch_size=EVAL_BATCH,
            )

            run_df.insert(0, "experiment", experiment)
            run_df.insert(1, "V", V)
            run_df.insert(2, "k", k)
            run_df.insert(3, "L", L)
            run_df.insert(4, "seed", seed)
            run_df["checkpoint"] = str(ckpt_path)
            run_df["run_dir"] = str(run_dir)

            all_rows.append(run_df)

            del model, P
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

raw = pd.concat(all_rows, ignore_index=True)
raw_path = CSV_DIR / "causal_intervention_seed_level.csv"
raw.to_csv(raw_path, index=False)

print("Saved:", raw_path)
display(raw)

## Seed-level bootstrap summaries

The statistical unit is the independent trained source/model seed. Contexts within a run are used only to estimate each run-level effect accurately.

For consistency with the manuscript, 95% intervals use 5,000 nonparametric bootstrap resamples over seeds with analysis seed 123. With five Part 1 seeds, these intervals should be interpreted as descriptive summaries of seed-level variation rather than high-precision population uncertainty.

In [ ]:
def bootstrap_mean_ci(values, reps=BOOTSTRAP_REPS, seed=BOOTSTRAP_SEED):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan, np.nan

    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(values), size=(reps, len(values)))
    boot = values[idx].mean(axis=1)
    return (
        float(values.mean()),
        float(np.quantile(boot, 0.025)),
        float(np.quantile(boot, 0.975)),
    )


summary_rows = []
metric_cols = [
    "tracking_gain_mean",
    "model_shift_js_mean",
    "oracle_shift_js_mean",
    "kl_target_before_mean",
    "kl_target_after_mean",
    "positive_tracking_fraction",
]

for (k, L, intervention), d in raw.groupby(["k", "L", "intervention"]):
    row = {
        "k": int(k),
        "L": int(L),
        "intervention": intervention,
        "n_seeds": int(d["seed"].nunique()),
    }
    for metric in metric_cols:
        mean, lo, hi = bootstrap_mean_ci(d[metric].values)
        row[f"{metric}_mean_over_seeds"] = mean
        row[f"{metric}_ci95_lo"] = lo
        row[f"{metric}_ci95_hi"] = hi
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows).sort_values(["k", "L", "intervention"])
summary_path = CSV_DIR / "causal_intervention_summary.csv"
summary.to_csv(summary_path, index=False)

print("Saved:", summary_path)
display(summary)

## Paired critical-vs-irrelevant contrast at \(L=k+1\)

This is the cleanest negative-control comparison because both interventions are applied to the same trained model and source seed. The critical intervention changes the true Markov state; the irrelevant intervention changes only the extra token outside that state.

In [ ]:
paired_rows = []

for k in KS:
    L = k + 1
    d = raw[raw["k"].eq(k) & raw["L"].eq(L)]

    crit = (
        d[d["intervention"].eq("critical")]
        .set_index("seed")["tracking_gain_mean"]
    )
    irr = (
        d[d["intervention"].eq("irrelevant")]
        .set_index("seed")["tracking_gain_mean"]
    )

    common = sorted(set(crit.index) & set(irr.index))
    diff = np.array([crit.loc[s] - irr.loc[s] for s in common], dtype=float)

    mean, lo, hi = bootstrap_mean_ci(diff)
    paired_rows.append({
        "k": k,
        "L": L,
        "n_seeds": len(common),
        "critical_minus_irrelevant_tracking_gain": mean,
        "ci95_lo": lo,
        "ci95_hi": hi,
        "all_seed_differences_positive": bool(np.all(diff > 0)),
    })

paired = pd.DataFrame(paired_rows)
paired_path = CSV_DIR / "causal_intervention_paired_contrasts.csv"
paired.to_csv(paired_path, index=False)

print("Saved:", paired_path)
display(paired)

## Layer-wise activation patching

This stage uses only \(L=k+1\), where a state-changing and an irrelevant-token counterfactual can be matched on the **same trained model, source kernel, and factual contexts**.

For each encoder layer \(\ell\):

1. run the factual and counterfactual inputs through layer \(\ell\);
2. replace only the **final-position residual vector** of the factual stream with the corresponding vector from the counterfactual stream;
3. continue through the remaining encoder layers and the unchanged native output head;
4. measure movement toward the appropriate exact oracle distribution.

The critical patch uses activations induced by changing the oldest token in the true Markov state. The negative-control patch uses activations induced by changing only the extra token outside the state. This is an internal intervention; it does not retrain or modify any checkpoint.


In [ ]:
def _causal_mask(L, device):
    return torch.triu(torch.ones((L, L), dtype=torch.bool, device=device), diagonal=1)


def _embed_input(model, x):
    L = x.shape[1]
    pos = torch.arange(L, device=x.device)[None, :]
    return model.token_emb(x) + model.pos_emb(pos)


def _run_encoder_layer(layer, h, mask):
    # TransformerEncoderLayer uses src_mask in supported PyTorch releases.
    return layer(h, src_mask=mask)


@torch.no_grad()
def encoder_states(model, x):
    """Return residual stream after every encoder layer."""
    h = _embed_input(model, x)
    mask = _causal_mask(x.shape[1], x.device)
    states = []
    for layer in model.encoder.layers:
        h = _run_encoder_layer(layer, h, mask)
        states.append(h)
    return states, mask


@torch.no_grad()
def probs_from_patched_state(model, factual_state, donor_state, mask, patch_layer):
    """Patch final-position residual after patch_layer, then run downstream layers."""
    h = factual_state.clone()
    h[:, -1, :] = donor_state[:, -1, :]
    for j in range(patch_layer + 1, len(model.encoder.layers)):
        h = _run_encoder_layer(model.encoder.layers[j], h, mask)
    logits = model.head(model.norm(h[:, -1, :]))
    return torch.softmax(logits.float(), dim=-1)


@torch.no_grad()
def evaluate_activation_patching_run(model, P, V, k, source_seed, n_contexts, batch_size):
    L = k + 1
    sampler_seed = 18_100_000 + 10_000 * k + source_seed
    intervention_seed = 19_200_000 + 10_000 * k + source_seed

    sampler = PersistentMarkovSampler(
        P=P, V=V, k=k, batch_size=batch_size,
        max_history=L, seed=sampler_seed, warmup_steps=CHAIN_WARMUP
    )
    gen = torch.Generator(device=DEVICE)
    gen.manual_seed(intervention_seed)

    rows = []
    seen = 0
    while seen < n_contexts:
        b = min(batch_size, n_contexts - seen)
        if b == batch_size:
            x, _ = sampler.sample(L)
        else:
            tmp = PersistentMarkovSampler(
                P=P, V=V, k=k, batch_size=b, max_history=L,
                seed=sampler_seed + seen + 17, warmup_steps=CHAIN_WARMUP
            )
            x, _ = tmp.sample(L)

        # Counterfactuals on identical factual contexts.
        x_crit = x.clone()
        critical_pos = L - k  # == 1 when L=k+1
        x_crit[:, critical_pos] = make_alternative_tokens(x[:, critical_pos], V, gen)

        x_irrel = x.clone()
        x_irrel[:, 0] = make_alternative_tokens(x[:, 0], V, gen)

        p_fact = oracle_probs(P, x, V, k)
        p_crit = oracle_probs(P, x_crit, V, k)
        p_irrel = oracle_probs(P, x_irrel, V, k)
        if (p_irrel - p_fact).abs().max().item() > 1e-7:
            raise AssertionError('Irrelevant activation-patching control changed oracle.')

        q_fact = model_probs(model, x)

        # Disable autocast here so cached/continued residual streams stay numerically consistent.
        fact_states, mask = encoder_states(model, x)
        crit_states, _ = encoder_states(model, x_crit)
        irrel_states, _ = encoder_states(model, x_irrel)

        for layer_idx in range(len(model.encoder.layers)):
            q_patch_crit = probs_from_patched_state(
                model, fact_states[layer_idx], crit_states[layer_idx], mask, layer_idx
            )
            q_patch_irrel = probs_from_patched_state(
                model, fact_states[layer_idx], irrel_states[layer_idx], mask, layer_idx
            )

            crit_gain = kl_pq(p_crit, q_fact) - kl_pq(p_crit, q_patch_crit)
            irr_gain = kl_pq(p_fact, q_fact) - kl_pq(p_fact, q_patch_irrel)

            rows.append(pd.DataFrame({
                'layer': layer_idx + 1,
                'critical_patch_gain': crit_gain.cpu().numpy(),
                'irrelevant_patch_gain': irr_gain.cpu().numpy(),
                'critical_patch_model_shift_js': js_divergence(q_fact, q_patch_crit).cpu().numpy(),
                'irrelevant_patch_model_shift_js': js_divergence(q_fact, q_patch_irrel).cpu().numpy(),
                'oracle_shift_js': js_divergence(p_fact, p_crit).cpu().numpy(),
            }))

        seen += b

    d = pd.concat(rows, ignore_index=True)
    out = []
    for layer, g in d.groupby('layer'):
        row = {'layer': int(layer), 'n_contexts': int(len(g))}
        for col in [
            'critical_patch_gain', 'irrelevant_patch_gain',
            'critical_patch_model_shift_js', 'irrelevant_patch_model_shift_js',
            'oracle_shift_js'
        ]:
            s = summarize_array(g[col].values)
            for stat, val in s.items():
                row[f'{col}_{stat}'] = val
        row['critical_minus_irrelevant_patch_gain_mean'] = float(
            g['critical_patch_gain'].mean() - g['irrelevant_patch_gain'].mean()
        )
        out.append(row)
    return pd.DataFrame(out)


In [ ]:
patch_rows = []

if RUN_ACTIVATION_PATCHING:
    for k in KS:
        experiment = EXPERIMENT_BY_K[k]
        L = k + 1
        for seed in SEEDS:
            print(f"PATCH k={k} L={L} seed={seed} [{experiment}]")
            model, P, cfg, run_dir, ckpt_path = load_model_and_kernel(
                experiment=experiment, V=V, k=k, L=L, seed=seed
            )
            d = evaluate_activation_patching_run(
                model=model, P=P, V=V, k=k, source_seed=seed,
                n_contexts=PATCH_N_CONTEXTS, batch_size=PATCH_BATCH
            )
            d.insert(0, 'experiment', experiment)
            d.insert(1, 'V', V)
            d.insert(2, 'k', k)
            d.insert(3, 'L', L)
            d.insert(4, 'seed', seed)
            d['checkpoint'] = str(ckpt_path)
            patch_rows.append(d)
            del model, P
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    patch_raw = pd.concat(patch_rows, ignore_index=True)
    patch_raw_path = CSV_DIR / 'activation_patching_seed_level.csv'
    patch_raw.to_csv(patch_raw_path, index=False)
    print('Saved:', patch_raw_path)
    display(patch_raw)

    patch_summary_rows = []
    for (k, layer), d in patch_raw.groupby(['k', 'layer']):
        row = {'k': int(k), 'L': int(k+1), 'layer': int(layer), 'n_seeds': int(d.seed.nunique())}
        for metric in [
            'critical_patch_gain_mean',
            'irrelevant_patch_gain_mean',
            'critical_minus_irrelevant_patch_gain_mean',
            'critical_patch_model_shift_js_mean',
            'irrelevant_patch_model_shift_js_mean'
        ]:
            mean, lo, hi = bootstrap_mean_ci(d[metric].values)
            row[f'{metric}_mean_over_seeds'] = mean
            row[f'{metric}_ci95_lo'] = lo
            row[f'{metric}_ci95_hi'] = hi
        patch_summary_rows.append(row)

    patch_summary = pd.DataFrame(patch_summary_rows).sort_values(['k','layer'])
    patch_summary_path = CSV_DIR / 'activation_patching_summary.csv'
    patch_summary.to_csv(patch_summary_path, index=False)
    print('Saved:', patch_summary_path)
    display(patch_summary)


In [ ]:
if RUN_ACTIVATION_PATCHING:
    fig, ax = plt.subplots(figsize=(8.5, 5.5))
    xpos = 0
    ticks, labels = [], []
    for k in KS:
        d = patch_summary[patch_summary.k == k].sort_values('layer')
        for _, r in d.iterrows():
            mean = r['critical_minus_irrelevant_patch_gain_mean_mean_over_seeds']
            lo = r['critical_minus_irrelevant_patch_gain_mean_ci95_lo']
            hi = r['critical_minus_irrelevant_patch_gain_mean_ci95_hi']
            ax.errorbar([xpos], [mean], yerr=[[mean-lo],[hi-mean]], fmt='o', capsize=4)
            ticks.append(xpos)
            labels.append(f"k={k}\nlayer {int(r['layer'])}")
            xpos += 1
    ax.axhline(0, linewidth=1)
    ax.set_xticks(ticks)
    ax.set_xticklabels(labels)
    ax.set_ylabel('Critical − irrelevant activation-patch gain (nats)')
    ax.set_title('Layer-wise causal steering of native prediction')
    fig.tight_layout()
    patch_png = FIG_DIR / 'FigC_activation_patching.png'
    patch_pdf = FIG_DIR / 'FigC_activation_patching.pdf'
    fig.savefig(patch_png, dpi=300)
    fig.savefig(patch_pdf)
    plt.show()
    print(patch_png)
    print(patch_pdf)


## Publication figures

Figure A reports the primary causal tracking metric. Figure B reports how much the model distribution itself moves under critical versus irrelevant interventions.

Error bars are descriptive seed-bootstrap 95% intervals.

In [ ]:
# Figure A — counterfactual oracle tracking gain
plot_rows = []
for (k, L, intervention), d in raw.groupby(["k", "L", "intervention"]):
    mean, lo, hi = bootstrap_mean_ci(d["tracking_gain_mean"].values)
    plot_rows.append({
        "k": k, "L": L, "intervention": intervention,
        "mean": mean, "lo": lo, "hi": hi,
    })
plot_df = pd.DataFrame(plot_rows)

fig, ax = plt.subplots(figsize=(8.5, 5.5))

x_labels, x, y, loerr, hierr = [], [], [], [], []

for k in KS:
    for L in [k, k + 1]:
        r = plot_df[
            (plot_df.k == k) &
            (plot_df.L == L) &
            (plot_df.intervention == "critical")
        ].iloc[0]
        x.append(len(x))
        x_labels.append(f"k={k}\nL={L}\ncritical")
        y.append(r["mean"])
        loerr.append(r["mean"] - r["lo"])
        hierr.append(r["hi"] - r["mean"])

    r = plot_df[
        (plot_df.k == k) &
        (plot_df.L == k + 1) &
        (plot_df.intervention == "irrelevant")
    ].iloc[0]
    x.append(len(x))
    x_labels.append(f"k={k}\nL={k+1}\nirrelevant")
    y.append(r["mean"])
    loerr.append(r["mean"] - r["lo"])
    hierr.append(r["hi"] - r["mean"])

ax.errorbar(x, y, yerr=np.vstack([loerr, hierr]), fmt="o", capsize=4)
ax.axhline(0, linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(x_labels)
ax.set_ylabel("KL reduction toward target oracle (nats)")
ax.set_title("Native prediction tracks state-changing counterfactuals")
fig.tight_layout()

fig_a_png = FIG_DIR / "FigA_causal_tracking_gain.png"
fig_a_pdf = FIG_DIR / "FigA_causal_tracking_gain.pdf"
fig.savefig(fig_a_png, dpi=300)
fig.savefig(fig_a_pdf)
plt.show()

print(fig_a_png)
print(fig_a_pdf)

In [ ]:
# Figure B — model-distribution shift
shift_rows = []
for (k, L, intervention), d in raw.groupby(["k", "L", "intervention"]):
    mean, lo, hi = bootstrap_mean_ci(d["model_shift_js_mean"].values)
    shift_rows.append({
        "k": k, "L": L, "intervention": intervention,
        "mean": mean, "lo": lo, "hi": hi,
    })
shift_df = pd.DataFrame(shift_rows)

fig, ax = plt.subplots(figsize=(8.5, 5.5))

x_labels, x, y, loerr, hierr = [], [], [], [], []

for k in KS:
    L = k + 1
    for intervention in ["critical", "irrelevant"]:
        r = shift_df[
            (shift_df.k == k) &
            (shift_df.L == L) &
            (shift_df.intervention == intervention)
        ].iloc[0]
        x.append(len(x))
        x_labels.append(f"k={k}\n{intervention}")
        y.append(r["mean"])
        loerr.append(r["mean"] - r["lo"])
        hierr.append(r["hi"] - r["mean"])

ax.errorbar(x, y, yerr=np.vstack([loerr, hierr]), fmt="o", capsize=4)
ax.axhline(0, linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(x_labels)
ax.set_ylabel("Jensen–Shannon divergence of model predictions (nats)")
ax.set_title("State-changing vs. irrelevant-token intervention")
fig.tight_layout()

fig_b_png = FIG_DIR / "FigB_causal_model_shift_control.png"
fig_b_pdf = FIG_DIR / "FigB_causal_model_shift_control.pdf"
fig.savefig(fig_b_png, dpi=300)
fig.savefig(fig_b_pdf)
plt.show()

print(fig_b_png)
print(fig_b_pdf)

## Validation checks and interpretation limits

The experiment is considered technically valid only if:

1. all 30 requested checkpoints/kernels are found;
2. irrelevant-token interventions leave the exact oracle unchanged;
3. all seed-level outputs are finite;
4. each \(L=k+1\) condition contains paired critical and irrelevant results for all five seeds;
5. when activation patching is enabled, every \(k\), seed, and encoder layer has a finite matched critical/control result.

Interpretation is based on the reported effect sizes, layer profiles, matched controls, and seed-bootstrap intervals; the intervention results are not treated as evidence for a uniquely identified internal circuit.


In [ ]:
expected_critical = len(KS) * 2 * len(SEEDS)
expected_irrelevant = len(KS) * len(SEEDS)

n_critical = int((raw["intervention"] == "critical").sum())
n_irrelevant = int((raw["intervention"] == "irrelevant").sum())

assert n_critical == expected_critical, (n_critical, expected_critical)
assert n_irrelevant == expected_irrelevant, (n_irrelevant, expected_irrelevant)

numeric = raw.select_dtypes(include=[np.number])
assert np.isfinite(numeric.to_numpy()).all(), "Non-finite numeric output detected."

for k in KS:
    d = raw[(raw.k == k) & (raw.L == k + 1)]
    cseeds = set(d[d.intervention == "critical"].seed)
    iseeds = set(d[d.intervention == "irrelevant"].seed)
    assert cseeds == set(SEEDS)
    assert iseeds == set(SEEDS)


if RUN_ACTIVATION_PATCHING:
    expected_patch_rows = len(KS) * len(SEEDS) * 3  # 3 encoder layers in selected runs
    assert len(patch_raw) == expected_patch_rows, (len(patch_raw), expected_patch_rows)
    patch_num = patch_raw.select_dtypes(include=[np.number])
    assert np.isfinite(patch_num.to_numpy()).all(), 'Non-finite activation-patching output detected.'
    for k in KS:
        dk = patch_raw[patch_raw.k == k]
        assert set(dk.seed) == set(SEEDS)
        assert set(dk.layer) == {1, 2, 3}

audit = {
    "status": "PASS",
    "V": V,
    "k_values": KS,
    "seeds": SEEDS,
    "n_contexts_per_model": N_CONTEXTS,
    "critical_seed_level_rows": n_critical,
    "irrelevant_seed_level_rows": n_irrelevant,
    "bootstrap_reps": BOOTSTRAP_REPS,
    "bootstrap_seed": BOOTSTRAP_SEED,
    "activation_patching_enabled": RUN_ACTIVATION_PATCHING,
    "patch_contexts_per_model": PATCH_N_CONTEXTS if RUN_ACTIVATION_PATCHING else 0,
    "claim_guardrail": (
        "Input intervention supports functional dependence on state-defining input; "
        "activation patching tests whether internal residual-stream information induced "
        "by the state change can causally steer the native prediction. Does not establish "
        "causal use of pretrained probe representations, a unique code, or a unique circuit."
    ),
}

audit_path = OUT_DIR / "causal_intervention_audit.json"
audit_path.write_text(json.dumps(audit, indent=2), encoding="utf-8")

print(json.dumps(audit, indent=2))
print("\nAll checks passed.")

## Exported outputs

After `Run All`, send back the generated `Part1_CAUSAL_INTERVENTION_RESULTS.zip`. It will include the input-intervention CSVs/figures, the activation-patching CSVs/figure, and the audit JSON.

Do not revise the manuscript claim until these outputs have been inspected.


In [ ]:
# Optional: zip only the NEW causal-intervention outputs.
import zipfile

zip_path = PART1_ROOT / "Part1_CAUSAL_INTERVENTION_RESULTS.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in OUT_DIR.rglob("*"):
        if file.is_file():
            z.write(file, file.relative_to(OUT_DIR))

print("Created:", zip_path)